In [49]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

In [50]:
df = pd.read_csv("../data/raw/sri_lanka_survey.csv")
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Loaded: 120 rows × 29 columns
Columns: ['Timestamp', '1. What is your gender?', '2. What is your age range?', '3. Which district are you from?', '4. What is your occupation?', '5. What is your approximate monthly household spending on products? (LKR)', '6. How often do you shop for products?', '06. Does your cultural background influence what products you buy?', 'Q07.  Clothing & Apparel\nWhat do you mainly check before buying a clothing & apparel product?', 'Q08. Clothing & Apparel  \nWhat is your main reason for buying clothing & apparel products?', 'Q09. Clothing & Apparel   \nHow much does emotional appeal (how it makes you feel) influence your purchase of clothing & apparel products?', 'Q10.   Beauty & Personal Care\nWhat do you mainly check before buying a beauty & personal care product?', 'Q11.   Beauty & Personal Care\n What is your main reason for buying beauty & personal care products?', 'Q12.   Beauty & Personal Care\nHow much does emotional appeal (how it makes you feel) in

In [51]:
COL_GENDER    = '1. What is your gender?'
COL_AGE       = '2. What is your age range?'
COL_DISTRICT  = '3. Which district are you from?'
COL_OCC       = '4. What is your occupation?'
COL_SPENDING  = '5. What is your approximate monthly household spending on products? (LKR)'
COL_CULTURE   = '06. Does your cultural background influence what products you buy?'

In [52]:
SCALE_COLS = [
    'Q09. Clothing & Apparel   \nHow much does emotional appeal (how it makes you feel) influence your purchase of clothing & apparel products?',
    'Q12.   Beauty & Personal Care\nHow much does emotional appeal (how it makes you feel) influence your purchase of beauty & personal care products?',
    'Q15.  Electronics\nHow much does emotional appeal (how it makes you feel) influence your purchase of electronics products?',
    'Q18.    Grocery & Food  \n  How much does emotional appeal (how it makes you feel) influence your purchase of grocery & food products?',
    'Q21.  Baby Products\nHow much does emotional appeal (how it makes you feel) influence your purchase of baby products products?',
    'Q24.  Pet Products\nHow much does emotional appeal (how it makes you feel) influence your purchase of pet products products?',
    'Q27.  Sports & Fitness\nHow much does emotional appeal (how it makes you feel) influence your purchase of sports & fitness products?',
]

In [53]:
REASON_COLS = [
    'Q08. Clothing & Apparel  \nWhat is your main reason for buying clothing & apparel products?',
    'Q11.   Beauty & Personal Care\n What is your main reason for buying beauty & personal care products?',
    'Q14.  Electronics \nWhat is your main reason for buying electronics products?',
    'Q17.    Grocery & Food\nWhat is your main reason for buying grocery & food products?',
    'Q20. Baby Products\nWhat is your main reason for buying baby products products?',
    'Q23.  Pet Products\nWhat is your main reason for buying pet products products?',
    'Q26.  Sports & Fitness\nWhat is your main reason for buying sports & fitness products?',
]

In [54]:
CHECK_COLS = [
    'Q07.  Clothing & Apparel\nWhat do you mainly check before buying a clothing & apparel product?',
    'Q10.   Beauty & Personal Care\nWhat do you mainly check before buying a beauty & personal care product?',
    'Q13.  Electronics\nWhat do you mainly check before buying a electronics product?',
    'Q16.   Grocery & Food\nWhat do you mainly check before buying a grocery & food product?',
    'Q19.  Baby Products \n  What do you mainly check before buying a baby products product?',
    'Q22.  Pet Products\nWhat do you mainly check before buying a pet products product?',
    'Q25.  Sports & Fitness\nWhat do you mainly check before buying a sports & fitness product?',
]

In [55]:
all_expected = [COL_GENDER, COL_AGE, COL_DISTRICT, COL_OCC,
                COL_SPENDING, COL_CULTURE] + SCALE_COLS + REASON_COLS + CHECK_COLS
 
missing = [c for c in all_expected if c not in df.columns]
if missing:
    print(f"WARNING: {len(missing)} columns not found: {missing[:3]}...")
else:
    print("All expected columns found ")

All expected columns found 


In [56]:
#encoders for demographic features
def enc_gender(v):
    v = str(v).strip().lower()
    return 1.0 if 'female' in v else 0.0

In [57]:
def enc_age(v):
    m = {'under 18':0,'18 – 24':1,'25 – 34':2,
         '35 – 44':3,'45 – 54':4,'55 and above':5}
    return float(m.get(str(v).strip(), 2))

In [58]:
def enc_district(d):
    urban = ['colombo','gampaha','kandy','galle','kalutara',
             'kurunegala','ratnapura','matara','jaffna']
    return 1.0 if str(d).lower().strip() in urban else 0.0

In [59]:
def enc_occupation(v):
    m = {
        'student': 0.7,
        'homemaker': 0.5,
        'private sector employee': 0.4,
        'self-employed/business owner': 0.3,
        'other': 0.5,
    }
    return m.get(str(v).strip().lower(), 0.5)

In [60]:
def enc_spending(v):
    m = {
        'less than rs. 5,000':    0.0,
        'rs. 5,000 – rs. 15,000': 0.25,
        'rs. 15,001 – rs. 30,000':0.5,
        'rs. 30,001 – rs. 50,000':0.75,
        'more than rs. 50,000':   1.0,
    }
    return m.get(str(v).strip().lower(), 0.5)

In [61]:
def enc_culture(v):
    v = str(v).strip().lower()
    if 'strongly' in v: return 1.0
    if 'no' in v:       return 0.0
    return 0.5 

In [62]:
#apply for all 
df['gender_enc']      = df[COL_GENDER].apply(enc_gender)
df['age_enc']         = df[COL_AGE].apply(enc_age)
df['environment_enc'] = df[COL_DISTRICT].apply(enc_district)
df['occupation_enc']  = df[COL_OCC].apply(enc_occupation)
df['spending_enc']    = df[COL_SPENDING].apply(enc_spending)
df['culture_enc']     = df[COL_CULTURE].apply(enc_culture)
 
print("Demographic features encoded ")

Demographic features encoded 


In [63]:
scale_matrix = pd.DataFrame(index=df.index)
for col in SCALE_COLS:
    scale_matrix[col] = pd.to_numeric(df[col], errors='coerce').fillna(3)

In [64]:
df['avg_appeal_raw']    = scale_matrix.mean(axis=1)           # 1–5
df['avg_appeal_norm']   = df['avg_appeal_raw'] - 3            # -2 to +2
df['high_appeal_count'] = (scale_matrix >= 4).sum(axis=1)     # num categories ≥ 4
df['low_appeal_count']  = (scale_matrix <= 2).sum(axis=1)     # num categories ≤ 2
 
print(f"\nAvg emotional appeal stats:")
print(df['avg_appeal_raw'].describe().round(2))


Avg emotional appeal stats:
count    120.00
mean       3.58
std        0.50
min        2.43
25%        3.29
50%        3.57
75%        3.89
max        4.86
Name: avg_appeal_raw, dtype: float64


In [68]:

def is_emotional_reason(v):
    return 1 if 'emotionally' in str(v).lower() else 0

def is_rational_reason(v):
    return 1 if ('pratical' in str(v).lower() or 'value for money' in str(v).lower()) else 0

In [69]:
#create new columns for emotional and rational reasons
emo_reason_cols = []
rat_reason_cols = []
for col in REASON_COLS: 
    e_col = col + '_emo'
    r_col = col + '_rat'
    df[e_col] = df[col].apply(is_emotional_reason)
    df[r_col] = df[col].apply(is_rational_reason)
    emo_reason_cols.append(e_col)
    rat_reason_cols.append(r_col)
    

df['emotional_reason_count'] = df[emo_reason_cols].sum(axis=1) 
df['rational_reason_count']  = df[rat_reason_cols].sum(axis=1) 

print(f"\nEmotional reason count distribution:")
print(df['emotional_reason_count'].value_counts().sort_index())


Emotional reason count distribution:
emotional_reason_count
0    18
1    37
2    35
3    23
4     5
5     2
Name: count, dtype: int64


In [70]:
rational_signal = ['specifications', 'ingredients', 'customer reviews']
emotional_signal = ['appearance', 'design']

In [71]:
#function to count signals in a given text
def count_signals(val, signals):
    v = str(val).lower()
    return sum(1 for s in signals if s in v)

In [72]:
rational_check_cols = []
emotional_check_cols = []

for col in CHECK_COLS:
    r_col = col + '_rat_signal'
    e_col = col + '_emo_signal'
    df[r_col] = df[col].apply(lambda x: count_signals(x, rational_signal))
    df[e_col] = df[col].apply(lambda x: count_signals(x, emotional_signal))
    rational_check_cols.append(r_col)
    emotional_check_cols.append(e_col)

In [73]:
df['rational_check_total']   = df[rational_check_cols].sum(axis=1)
df['emotional_check_total']  = df[emotional_check_cols].sum(axis=1)

print(f"\nRational check total stats:")
print(df['rational_check_total'].describe().round(2))


Rational check total stats:
count    120.00
mean      11.39
std        3.35
min        2.00
25%        9.00
50%       11.50
75%       14.00
max       18.00
Name: rational_check_total, dtype: float64


In [74]:
df['total_score'] = (
    df['avg_appeal_norm']         * 3.0 +   # most reliable signal 
    df['emotional_reason_count']  * 2.0 +   # explicit emotional reason
    df['high_appeal_count']       * 1.0 -   # many high-appeal categories
    df['rational_reason_count']   * 2.0 -   # explicit rational reason
    df['rational_check_total']    * 0.5     # checking specs/reviews
)

In [75]:
print(f"\nTotal score stats:")
print(df['total_score'].describe().round(3))


Total score stats:
count    120.000
mean       0.442
std        5.482
min      -13.857
25%       -2.714
50%        0.500
75%        4.589
max       15.571
Name: total_score, dtype: float64


In [ ]:
#cognitive label
median_score = df['total_score'].median()
df['cognitive_label'] = (df['total_score'] > median_score).astype(int)

In [79]:
print(f"\nMedian score: {median_score:.3f}")
print(f"System 1 (emotional): {(df.cognitive_label==1).sum()}")
print(f"System 2 (rational):  {(df.cognitive_label==0).sum()}")


Median score: 0.500
System 1 (emotional): 60
System 2 (rational):  60


In [80]:
print(df[['1. What is your gender?','2. What is your age range?',
          'avg_appeal_raw','emotional_reason_count',
          'total_score','cognitive_label']].to_string())

    1. What is your gender? 2. What is your age range?  avg_appeal_raw  emotional_reason_count  total_score  cognitive_label
0                    Female                    18 – 24        4.285714                       2     2.857143                1
1                      Male                    45 – 54        3.714286                       1     5.142857                1
2                      Male                    35 – 44        4.285714                       0    -3.142857                0
3                    Female                    25 – 34        4.142857                       4    11.428571                1
4                      Male                    18 – 24        4.571429                       3     6.714286                1
5                      Male                    18 – 24        4.285714                       2     0.357143                0
6                      Male                    18 – 24        3.285714                       3     1.357143                1


In [81]:
FEATURE_COLS = [
    'gender_enc',            # From Q1
    'age_enc',               # From Q2
    'environment_enc',       # From Q3 (urban/rural)
    'occupation_enc',        # From Q4
    'spending_enc',          # From Q5
    'culture_enc',           # From Q06
    'avg_appeal_norm',       # Average of 7 emotional appeal scales (normalized)
    'emotional_reason_count',# Count of emotional reason answers (0–7)
    'rational_reason_count', # Count of rational reason answers (0–7)
    'rational_check_total',  # Count of rational check signals across 7 categories
    'emotional_check_total', # Count of emotional check signals across 7 categories
]

In [83]:
print(f"\nFeature columns ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)
 
X = df[FEATURE_COLS].values
y = df['cognitive_label'].values


Feature columns (11):
['gender_enc', 'age_enc', 'environment_enc', 'occupation_enc', 'spending_enc', 'culture_enc', 'avg_appeal_norm', 'emotional_reason_count', 'rational_reason_count', 'rational_check_total', 'emotional_check_total']


In [84]:
#train model 

model= {
    "LogisticRegression":LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,C=0.5),
    "RandomForest":RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=4, min_samples_leaf=2 ,random_state=42),
    "GradientBoosting":GradientBoostingClassifier(n_estimators=50,max_depth=2,learning_rate=0.1, random_state=42),
    "SVC":SVC(kernel='rbf', class_weight='balanced',probability=True, random_state=42),
}

In [ ]:
from sklearn.model_selection import LeaveOneOut
# use Leave-One-Out Cross-Validation to evaluate each model
loo = LeaveOneOut()
results = {}
for name, estimator in model.items():
    estimator.fit(X, y)

    loo_preds = cross_val_score(estimator, X, y, cv=loo, scoring='accuracy')
    loo_f1 = cross_val_score(estimator, X, y, cv=loo, scoring='f1_weighted')

    results[name] = {
        'accuracy': loo_preds.mean(),
        'f1_weighted': loo_f1.mean(),
        'model': estimator
    }

    print(f"\n{name}:")
    print(f"  LOO Accuracy: {loo_preds.mean():.4f} ± {loo_preds.std():.4f}")
    print(f"  LOO F1:       {loo_f1.mean():.4f} ± {loo_f1.std():.4f}")


LogisticRegression:
  LOO Accuracy: 0.8833 ± 0.3210
  LOO F1:       0.8833 ± 0.3210

RandomForest:
  LOO Accuracy: 0.8750 ± 0.3307
  LOO F1:       0.8750 ± 0.3307

GradientBoosting:
  LOO Accuracy: 0.8583 ± 0.3487
  LOO F1:       0.8583 ± 0.3487

SVC:
  LOO Accuracy: 0.8167 ± 0.3869
  LOO F1:       0.8167 ± 0.3869


In [88]:
best_name = max(results, key=lambda k: results[k]['f1_weighted'])
best_model = results[best_name]['model']
print(f"\nBest model: {best_name}")


Best model: LogisticRegression


In [89]:
if hasattr(best_model, 'feature_importances_'):
    for feat, imp in sorted(zip(FEATURE_COLS, best_model.feature_importances_),
                            key=lambda x: -x[1]):
        bar = '█' * max(1, int(imp * 40))
        print(f"  {feat:<28} {bar} {imp:.3f}")

In [91]:
joblib.dump(best_model,   "../models/demographic_classifier.pkl")
joblib.dump(FEATURE_COLS, "../models/demographic_feature_cols.pkl")

['../models/demographic_feature_cols.pkl']

In [92]:
df[FEATURE_COLS + ['cognitive_label']].to_csv(
    "../data/processed/demographic_features.csv", index=False
)

In [93]:
df.to_csv("../data/processed/survey_labeled.csv", index=False)

In [94]:
print(f"\nNote: Dataset has only {len(df)} responses.")


Note: Dataset has only 120 responses.
